# 3D Geological Model

Interactive construction and visualization of the geological model using private interface-point and orientation observations. The main scripted workflow is described in the repository README.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent / "scripts"))
from project_paths import required_file, output_path


In [ ]:
%matplotlib inline
from pyvista import set_plot_theme

set_plot_theme('document')
import numpy as np
import pandas as pd
import os
import time


In [ ]:

np.random.seed(1515)
import gempy as gp
import gempy_viewer as gpv
import pyvista as pv
import matplotlib.pyplot as plt
import copy
import pickle

print("Notebook stage 3: processing")
start_time = time.time()



geo_model: gp.data.GeoModel = gp.create_geomodel(
    project_name='Fankou',
    extent=[462950, 464050, 2777200, 2779000, -800, 100],
    resolution=[32, 32, 32],
    importer_helper=gp.data.ImporterHelper(
        path_to_orientations=required_file("orientations.csv"),
        path_to_surface_points=required_file("points.csv"),
    )
)
print("Notebook stage 3: processing")

print("Notebook stage 3: processing")
gp.map_stack_to_surfaces(
    gempy_model=geo_model,
    mapping_object={
        "Fault_Series": ('F5', 'F6', 'F7'),
        "Strat_Series": ('C2+3ht', 'D3tc', 'D3tb', 'D3ta', 'D2db', 'D2da'),
    }
)

print("Notebook stage 3: processing")
gp.set_is_fault(
    frame=geo_model.structural_frame,
    fault_groups=['Fault_Series']
)


geo_model.interpolation_options.kernel_options.range = 17
geo_model.interpolation_options.kernel_options.nugget = 0.015

print("Notebook stage 3: processing")
gp.compute_model(geo_model)
print("Notebook stage 3: processing")





In [ ]:
pv.set_plot_theme('document')

print("Notebook stage 4: processing")

view_3d = gpv.plot_3d(
    geo_model,
    show_data=False,
    image=False,
    show=False,
    kwargs_plot_structured_grid={
        'opacity': 0,        
        'show_edges': False,    
        # 'cmap'   -
    }
)

p = view_3d.p


p.set_background('white')
p.hide_axes()
p.remove_bounds_axes()


p.camera.azimuth = 45
p.camera.elevation = 25
p.camera.zoom(1.2)


p.window_size = [2400, 1800]


out_dir = './figures'
os.makedirs(out_dir, exist_ok=True)
out_overall = os.path.join(out_dir, 'model_3D_overall.png')

p.show(screenshot=out_overall, title='FanKou Geology Model')

print("Notebook stage 4: processing")



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print("Visualizing Deterministic Model (Y-axis profile)...")

try:
    
    raw_lith_block = geo_model.solutions.raw_arrays.lith_block

    
    grid_shape = geo_model.grid.regular_grid.resolution # (res_x, res_y, res_z)
    extent = geo_model.grid.regular_grid.extent         # (min_x, max_x, min_y, max_y, min_z, max_z)

    res_x, res_y, res_z = grid_shape

    
    lith_block_3d = raw_lith_block.reshape(grid_shape)
    print(f"Reshaped block to 3D: {lith_block_3d.shape}")

    # 2.   (Y section )
    
    mid_y_idx = 0

    
    
    profile_2d = lith_block_3d[:, mid_y_idx, :]

    print(f"Extracted Y-profile at index {mid_y_idx} (Y coord approx: {(extent[2] + extent[3])/2})")

    
    plt.figure(figsize=(12, 8))

    # imshow
    # origin='lower'   Z
    # extent   [x_min, x_max, z_min, z_max]
    im = plt.imshow(
        profile_2d.T,  
        origin='lower',
        extent=[extent[0], extent[1], extent[4], extent[5]],
        cmap='viridis', 
        aspect='auto'   
    )

    
    plt.title(f"Deterministic Model Profile (Y-axis Slice at Index {mid_y_idx})", fontsize=14)
    plt.xlabel("X Coordinate (m)", fontsize=12)
    plt.ylabel("Z Elevation (m)", fontsize=12)

    
    cbar = plt.colorbar(im, label="Rock ID / Lithology Index")

    
    plt.tight_layout()
    plt.savefig('deterministic_model_Y_profile.png', dpi=300)
    plt.show()

    print("Y-axis profile saved as 'deterministic_model_Y_profile.png'")

except Exception as e:
    print(f"Visualization failed: {e}")
    import traceback
    traceback.print_exc()


In [ ]:
view_3d = gpv.plot_3d(
    geo_model,
    show_data=True,
    image=False,
    kwargs_plot_structured_grid={'opacity': 0},
    show=False
)



view_3d.p.show(screenshot=output_path('deterministic_model_3D.png'))
print("3D ")


In [ ]:

print("Notebook stage 8: processing")
mc_start_time = time.time()


NUM_SIMULATIONS = 3  
STD_DEV_Z_STRATA = 5.0  
STD_DEV_X_FAULT = 10.0  


original_surface_points = geo_model.surface_points_copy.df.copy()
original_orientations = geo_model.orientations_copy.df.copy()


surface_info = {}
for element in geo_model.structural_frame.structural_elements:
    if hasattr(element, 'surfaces'):
        for surface in element.surfaces:
            surface_info[surface.name] = element.name


original_surface_points['surface'] = original_surface_points.index.map(
    lambda x: next((k for k, v in surface_info.items() if str(x) in v), None)
)
original_surface_points['element'] = original_surface_points['surface'].map(surface_info)


original_orientations['surface'] = original_orientations.index.map(
    lambda x: next((k for k, v in surface_info.items() if str(x) in v), None)
)
original_orientations['element'] = original_orientations['surface'].map(surface_info)


simulation_results = []


for sim in range(NUM_SIMULATIONS):
    sim_start_time = time.time()

    try:
        print("Notebook stage 8: processing")

        # GeoModel  -
        sim_model = gp.create_geomodel(
            project_name=f'Fankou_MC_{sim + 1}',
            extent=[462950, 464050, 2777200, 2779000, -800, 100],
            resolution=[32, 32, 32],
            structural_frame=copy.deepcopy(geo_model.structural_frame)
        )

        
        sim_model.interpolation_options = copy.deepcopy(geo_model.interpolation_options)

        
        surface_points_df = original_surface_points.copy()
        orientations_df = original_orientations.copy()

        
        strat_mask = surface_points_df['element'] == 'Strat_Series'
        z_perturbations = np.random.normal(0, STD_DEV_Z_STRATA, size=np.sum(strat_mask))
        surface_points_df.loc[strat_mask, 'Z'] += z_perturbations

        
        fault_mask = surface_points_df['element'] == 'Fault_Series'
        x_perturbations = np.random.normal(0, STD_DEV_X_FAULT, size=np.sum(fault_mask))
        surface_points_df.loc[fault_mask, 'X'] += x_perturbations

        
        for element in sim_model.structural_frame.structural_elements:
            element_mask = surface_points_df['element'] == element.name
            element_data = surface_points_df[element_mask]

            if len(element_data) > 0:
                element.surface_points.set_data(
                    x=element_data['X'].values,
                    y=element_data['Y'].values,
                    z=element_data['Z'].values
                )

        
        for element in sim_model.structural_frame.structural_elements:
            element_mask = orientations_df['element'] == element.name
            element_data = orientations_df[element_mask]

            if len(element_data) > 0:
                element.orientations.set_data(
                    x=element_data['X'].values,
                    y=element_data['Y'].values,
                    z=element_data['Z'].values,
                    G_x=element_data['G_x'].values,
                    G_y=element_data['G_y'].values,
                    G_z=element_data['G_z'].values
                )

        
        print("Notebook stage 8: processing")
        gp.compute_model(sim_model)

        
        print("Notebook stage 8: processing")
        print(surface_points_df.head())

        
        simulation_results.append({
            'model': sim_model,
            'surface_points': surface_points_df.copy()
        })

        print("Notebook stage 8: processing")

    except Exception as e:
        print("Notebook stage 8: processing")
        continue

print("Notebook stage 8: processing")


In [ ]:
import pickle


with open(output_path('monte_carlo_results.pkl'), 'wb') as f:
    pickle.dump(simulation_results, f)

print("✅   monte_carlo_results.pkl")


In [ ]:
# ==========================================

# ==========================================
print("\n" + "=" * 50)
print("Notebook stage 10: processing")
analysis_start_time = time.time()


if not simulation_results:
    print("Notebook stage 10: processing")
else:
    
    print("Notebook stage 10: processing")

    
    grid_shape = geo_model.grid.regular_grid.resolution
    extent = geo_model.grid.regular_grid.extent
    n_cells = np.prod(grid_shape)

    
    all_rock_types = np.zeros((len(simulation_results), n_cells))

    for i, result in enumerate(simulation_results):
        try:
            block_data = result['model'].solutions.raw_arrays.lith_block
            all_rock_types[i, :] = block_data.flatten()
            print("Notebook stage 10: processing")
        except Exception as e:
            print("Notebook stage 10: processing")
            all_rock_types[i, :] = np.nan

    
    print("Notebook stage 10: processing")
    uncertainty = np.zeros(n_cells)
    valid_mask = ~np.isnan(all_rock_types).any(axis=0)
    valid_data = all_rock_types[:, valid_mask]

    for orig_idx in range(n_cells):
        if not valid_mask[orig_idx]:
            uncertainty[orig_idx] = np.nan
            continue

        valid_col_idx = np.where(valid_mask[:orig_idx + 1])[0][-1]
        values = valid_data[:, valid_col_idx]
        unique_vals, counts = np.unique(values, return_counts=True)
        max_count = np.max(counts)
        uncertainty[orig_idx] = 1 - (max_count / len(simulation_results))

    # ==========================================
    
    # ==========================================
    print("Notebook stage 10: processing")
    res_x, res_y, res_z = grid_shape
    mid_y_idx = res_y // 2

    
    try:
        original_2d = geo_model.solutions.raw_arrays.lith_block.reshape(grid_shape)[:, mid_y_idx, :]
    except:
        original_2d = np.zeros((res_x, res_z))

    
    sim_data_list = []
    for i in range(3):  
        if i < len(simulation_results):
            data = simulation_results[i]['model'].solutions.raw_arrays.lith_block.reshape(grid_shape)[:, mid_y_idx, :]
        else:
            data = original_2d  
        sim_data_list.append(data)

    
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))

    
    im0 = axes[0].imshow(original_2d.T, cmap="viridis", origin="lower",
                         extent=[extent[0], extent[1], extent[4], extent[5]])
    axes[0].set_title("0.  ")
    axes[0].set_xlabel("X")
    axes[0].set_ylabel("Z ( )")
    plt.colorbar(im0, ax=axes[0], label=" ID")

    
    for i in range(3):
        ax = axes[i + 1]
        im = ax.imshow(sim_data_list[i].T, cmap="viridis", origin="lower",
                       extent=[extent[0], extent[1], extent[4], extent[5]])
        ax.set_title(f"  {i + 1}")
        ax.set_xlabel("X")
        plt.colorbar(im, ax=ax, label=" ID")

    plt.tight_layout()
    plt.savefig('4_panel_comparison.png', dpi=200, bbox_inches='tight')
    plt.show()

    # ==========================================
    
    # ==========================================
    print("\n" + "-" * 40)
    print("Notebook stage 10: processing")

    # nan
    uncert_valid = uncertainty[valid_mask]

    
    mean_uncert = np.mean(uncert_valid)
    std_uncert = np.std(uncert_valid)

    
    high_uncert_threshold = 0.3
    high_uncert_ratio = np.sum(uncert_valid > high_uncert_threshold) / len(uncert_valid)

    
    if mean_uncert < 0.1:
        quality = "  ( )"
    elif mean_uncert < 0.2:
        quality = "  ( )"
    elif mean_uncert < 0.3:
        quality = "  ( )"
    else:
        quality = "  ( ， )"

    print("Notebook stage 10: processing")
    print("Notebook stage 10: processing")
    print("Notebook stage 10: processing")

    # ==========================================
    
    # ==========================================
    print("\n 3D ...")
    try:
        grid = pv.ImageData()
        grid.dimensions = [grid_shape[0] + 1, grid_shape[1] + 1, grid_shape[2] + 1]
        grid.origin = (extent[0], extent[2], extent[4])
        grid.spacing = (
            (extent[1] - extent[0]) / grid_shape[0],
            (extent[3] - extent[2]) / grid_shape[1],
            (extent[5] - extent[4]) / grid_shape[2]
        )

        uncertainty_clean = np.nan_to_num(uncertainty, nan=0.0)
        grid.cell_data["Uncertainty"] = uncertainty_clean

        p = pv.Plotter()
        p.add_mesh(grid, opacity=0.5, cmap="Reds", show_scalar_bar=True)
        p.add_title(f"3D   -  : {quality}")
        p.show(screenshot='3d_uncertainty_result.png')

    except Exception as e:
        print(f"3D : {e}")

    print(f"\n ！ : {time.time() - analysis_start_time:.2f} ")


In [ ]:
import xarray as xr


voxel_data_1 = xr.open_dataset(required_file('simulation_1_voxel.nc'))
voxel_data_2 = xr.open_dataset(required_file('simulation_2_voxel.nc'))


difference = voxel_data_1 - voxel_data_2


difference['lithology'].plot()
plt.title('Difference between Simulation 1 and Simulation 2')
plt.show()


In [ ]:
for i, result in enumerate(simulation_results):
    sim_model = result['model']
    
    grid = sim_model.grid.regular_grid
    x_s = np.linspace(grid.extent[0], grid.extent[1], grid.resolution[0] + 1)
    y_s = np.linspace(grid.extent[2], grid.extent[3], grid.resolution[1] + 1)
    z_s = np.linspace(grid.extent[4], grid.extent[5], grid.resolution[2] + 1)

    structured_grid = pv.StructuredGrid()
    x, y, z = np.meshgrid(x_s, y_s, z_s, indexing='ij')
    structured_grid.points = np.c_[x.ravel(order='F'), y.ravel(order='F'), z.ravel(order='F')]
    structured_grid.dimensions = [len(x_s), len(y_s), len(z_s)]

    
    lith_block = result['lith_block']
    lith_block_3d = lith_block.reshape(grid.resolution, order='F')
    structured_grid.cell_data['Lithology'] = lith_block_3d.ravel(order='F')

    # VTK
    structured_grid.save(f'simulation_{i + 1}.vtk')
    print("Notebook stage 12: processing")

print("Notebook stage 12: processing")


In [ ]:
print("Notebook stage 13: processing")

fig, axes = plt.subplots(2, min(5, NUM_SIMULATIONS), figsize=(15, 8))


if NUM_SIMULATIONS > 0:
    gpv.plot_2d(geo_model, ax=axes[0, 0], show_data=True, legend=False, show=False)
    axes[0, 0].set_title(' ')


for i in range(min(5, NUM_SIMULATIONS)):
    if i < len(simulation_results):
        gpv.plot_2d(
            simulation_results[i]['model'],
            ax=axes[1, i],
            show_data=True,
            legend=False,
            show=False
        )
        axes[1, i].set_title(f'  #{i + 1}')

plt.tight_layout()
plt.savefig(output_path('simulations_comparison.png'))
print("Notebook stage 13: processing")

print("Notebook stage 13: processing")
with open('monte_carlo_simulations.pkl', 'wb') as f:
    pickle.dump({
        'simulation_results': simulation_results,
        'original_model': geo_model,
        'simulation_parameters': {
            'NUM_SIMULATIONS': NUM_SIMULATIONS,
            'STD_DEV_Z_STRATA': STD_DEV_Z_STRATA,
            'STD_DEV_X_FAULT': STD_DEV_X_FAULT,
            'STD_DEV_ORIENTATION': STD_DEV_ORIENTATION
        }
    }, f)

print("Notebook stage 13: processing")
print("Notebook stage 13: processing")
